# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravikiranbathe/flyrank-ai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q duckdb huggingface_hub pyarrow pandas

In [2]:
from google.colab import userdata
from huggingface_hub import login
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

con = duckdb.connect()

In [3]:
con.sql("""
INSTALL httpfs;
LOAD httpfs;
INSTALL parquet;
LOAD parquet;
""")

In [4]:
from huggingface_hub import hf_hub_download

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print(march_path)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [5]:
con.sql(f"""
CREATE OR REPLACE VIEW fact_content_daily_performance AS
SELECT *
FROM read_parquet('{march_path}');
""")

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis

One row represents the observed daily search performance of one content item(webpage) for one client on one report date.

## Time window

THis notebook uses the March 2026 partition (month = 2026-03) to define the data contract and verify the warehouse data. The final month (June 2026) is treated as a sealed test period and is not used for feature development or label design.

In [6]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM fact_content_daily_performance
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘



In [7]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM fact_content_daily_performance;
""").show()

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

- gsc_clicks
- gsc_impressions
- gsc_sum_position
- sessions_ai

These are historical measurements or indicators that are available before making a content refresh decision and can support feature engineering.

## Label / Proxy

The model aims to rank content by refresh opportunity using a proxy target derived from historical performance. Label-derived fields are not used as features.

## Context

- client_hash_id
- content_hash_id
- report_date
- month
- ga4_data_available

These fields identify observations, support grouping and filtering, and provide context but are not used as model features.

## Excluded

- Future performance metrics
- Any label-derived fields
- Identifier fields (`client_hash_id`, `content_hash_id`) as model inputs

These fields are excluded because they either contain future information, introduce data leakage, or uniquely identify entities rather than describing content performance.

In [8]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_clicks,
    gsc_impressions,
    gsc_sum_position,
    ga4_data_available,
    sessions_ai,
    month
FROM fact_content_daily_performance
LIMIT 5;
""").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────┬─────────────────┬──────────────────┬────────────────────┬─────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_clicks │ gsc_impressions │ gsc_sum_position │ ga4_data_available │ sessions_ai │  month  │
│    date     │         varchar         │         varchar          │   int64    │      int64      │      int64       │      boolean       │    int64    │ varchar │
├─────────────┼─────────────────────────┼──────────────────────────┼────────────┼─────────────────┼──────────────────┼────────────────────┼─────────────┼─────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │          0 │              20 │               67 │ NULL               │        NULL │ 2026-03 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │          0 │               1 │                0 │ NULL               │        NULL │ 2026-03 │
│ 2026-03-01  │ 

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification

The following queries verify the data contract by checking the data grain, row counts, reporting window, and data availability for the selected March 2026 partition.

In [9]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
FROM fact_content_daily_performance;
""").show()

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int128       │       int128       │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

- Historical coverage differs across clients because Search Console and GA4 data begin on different dates.
- Rows where `ga4_data_available` is `FALSE` should not be interpreted as zero engagement because they indicate unavailable data rather than missing user activity.
- This notebook uses only the March 2026 partition, so it does not capture seasonal or long-term performance trends.
- The analysis provides observed, decision-support insights and does not establish causal relationships.

In [10]:
con.sql("""
SELECT
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS missing_gsc_clicks,
    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS missing_gsc_impressions,
    AVG(CASE WHEN ga4_data_available IS NULL THEN 1.0 ELSE 0 END) AS missing_ga4_flag
FROM fact_content_daily_performance;
""").show()

┌────────────────────┬─────────────────────────┬─────────────────────┐
│ missing_gsc_clicks │ missing_gsc_impressions │  missing_ga4_flag   │
│       double       │         double          │       double        │
├────────────────────┼─────────────────────────┼─────────────────────┤
│                0.0 │                     0.0 │ 0.30673966592889734 │
└────────────────────┴─────────────────────────┴─────────────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.